# RAFT-Large (C_T_SKHT_V2) — DIMER dense optical flow and bounded fine-tuning (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/raft-optical-flow-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/raft-optical-flow-pipeline/blob/main/tutorials/raft_optical_flow_colab.ipynb) [![torchvision](https://img.shields.io/badge/torchvision-raft__large-ee4c2c?style=flat&logo=pytorch&logoColor=white)](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.optical_flow.raft_large.html) [![Upstream](https://img.shields.io/badge/Upstream-pytorch%2Fvision-181717?style=flat&logo=github&logoColor=white)](https://github.com/pytorch/vision) [![arXiv](https://img.shields.io/badge/arXiv-2003.12039-b31b1b.svg)](https://arxiv.org/abs/2003.12039) [![License](https://img.shields.io/badge/License-Apache--2.0-green.svg)](https://github.com/kurtvalcorza/raft-optical-flow-pipeline/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** dense optical flow between two frames with torchvision's RAFT-Large, and a bounded fine-tune on frame pairs with known flow that is scored on a held-out split by end-point error against a zero-flow baseline and exported as a reloadable SafeTensors adapter

**This notebook is standalone.** It carries the repository's package (2 modules under `src/raft_optical_flow_pipeline/`, at revision `1d4a15b255ca`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and `download.pytorch.org` for the checkpoint file pinned by SHA-256 `ff5fadd56d26b40647388883af1547351ea17868b765c05b27231e72dd16a322` (~21 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (nothing is installed into the notebook's own Python, so no restart is needed and Run all completes in one pass), stages and digest-verifies the pinned checkpoint, estimates flow on a rendered pair with exact ground truth, validates the labelled dataset (40 pairs by default), splits it into training and held-out parts, measures the pre-adaptation baseline, **runs the bounded fine-tune**, re-evaluates on the held-out split, estimates flow on unseen pairs, exports the adapter, reloads it onto a fresh base model to verify the flow, and writes machine-readable outputs with provenance. Nothing is skipped behind a default-off flag, and no clone or DIMER worker is required (NOTEBOOK_SPEC 2.2 §5, RUN7, FT2).

**Bring Your Own Data:** Two optional BYOD branches are included, and both are off by default (`USE_BYOD_IMAGE = False`, `USE_BYOD_DATASET = False`). `USE_BYOD_IMAGE` runs your own two frames through the same validation, estimation and evaluation-report stages as the sample pair. `USE_BYOD_DATASET` takes your own frame pairs with ground-truth flow through the full adaptation workflow — validate, split, baseline, fine-tune, evaluate, export and reload — under NOTEBOOK_SPEC 2.2 DAT14. Set `BYOD_IMAGE1_PATH` and `BYOD_IMAGE2_PATH`, or `BYOD_DATASET_DIR`, to read from a location without an upload dialog (EXE2).

Optical flow is the apparent motion of every pixel between two frames: for each pixel `(x, y)` of frame 1, a displacement `(u, v)` in pixels such that the same scene point appears at `(x + u, y + v)` in frame 2. RAFT (Recurrent All-Pairs Field Transforms) computes it in three parts. A **feature encoder** turns each frame into features at 1/8 resolution; RAFT correlates every feature of frame 1 with every feature of frame 2 into a 4-D correlation volume; and a **recurrent update block** starts from zero flow and refines it a fixed number of times, looking up the volume around the current estimate at each step. A learned upsampler returns the flow at full resolution.

This notebook uses torchvision's `raft_large` builder with its default weights, `Raft_Large_Weights.C_T_SKHT_V2`. torchvision's weight documentation says these weights were trained from scratch on FlyingChairs and FlyingThings3D, then fine-tuned on a mix of Sintel, KITTI, HD1K and FlyingThings3D (clean pass), and lists Sintel test end-point errors of 1.819 (clean pass) and 3.067 (final pass).

**The default path really adapts the model:** it builds a labelled dataset of textured frame pairs with exact flow, measures the pretrained end-point error on a held-out split next to a zero-flow baseline, runs a bounded fine-tune with RAFT's own sequence loss, re-scores the held-out split, runs the adapted model on unseen pairs, exports the changed tensors as a SafeTensors adapter, and reloads that adapter onto a fresh copy of the verified base model to check that it reproduces the same flow. Every number you see is measured in this notebook runtime.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the pinned checkpoint; estimate flow on a frame pair with exact ground truth and score it by end-point error (EPE), outlier rates and angular error against the zero-flow baseline; see how the number of recurrent updates changes the estimate; probe the model with a static and a featureless pair; build and validate a labelled flow dataset; split it and measure a pre-adaptation baseline; run a bounded fine-tune with RAFT's sequence loss; score the adapted model on the held-out split; run inference on unseen pairs; and export, reload and verify the adapter.

**This notebook does not demonstrate:** real-video optical flow claims (the adaptation dataset is rendered in code with integer displacements, so the model learns these renderings and nothing about camera footage); Sintel or KITTI benchmark results (the metrics helper follows the benchmark definitions but is run on synthetic data only); full-schedule RAFT training (torchvision's reference recipe trains for hundreds of thousands of steps on several GPUs; the tutorial runs a few epochs on 30 pairs by default); stereo disparity; scene flow; video interpolation.

**Who this notebook is for.** The intended audience is a learner who knows basic Python, PIL and NumPy, has used Colab or Jupyter, and wants to see how a dense optical-flow model is scored against exact ground truth, where its estimates go wrong, and what a bounded fine-tune of its update block changes. No prior experience with RAFT or optical flow is assumed; terms are explained where they first matter and again in the **Glossary** at the end. A GPU runtime (T4) is the documented runtime; CPU works, slowly.

**Input → Model → Output.**

| | Flow estimation | Bounded fine-tune |
|---|---|---|
| Input | two frames of one size (PIL images) | rendered frame pairs with exact flow and validity masks (40 by default: 30 training, 10 held out) |
| Model | torchvision's RAFT-Large (`C_T_SKHT_V2`), a digest-pinned checkpoint, 12 recurrent updates | the same model; only the recurrent update block and upsampling mask predictor train (3.1 M of 5.3 M parameters), encoders and BatchNorm frozen |
| Output | an `(H, W, 2)` flow field in pixels; EPE, 1/3/5 px rates and angular error beside a zero-flow baseline | held-out EPE before and after, flow on unseen pairs, and a SafeTensors adapter that reloads to the same flow |

**How to use this notebook.** Choose a GPU runtime (**Runtime → Change runtime type → T4 GPU**), then **Runtime → Run all**. Run all completes in one pass: Section 1 installs nothing into the notebook's own Python, so no restart is needed. Sections 1–3 are **infrastructure** — the isolated environment, the carried package and the pinned checkpoint — and their cells are collapsed; you may run them without studying them. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited, and the defaults reproduce the recorded run. Before each principal result the notebook asks you to **Predict**; after it come **What to notice** and a collapsible **Check your reasoning** with a worked answer from the recorded run (the Kaggle Tesla T4 run of 25 September 2026 recorded in `docs/release-verification.md`; GPU kernels are not deterministic, so your last digits may differ). **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Writing your predictions down is optional.

**Roadmap:** 1–3 infrastructure → 4 the pretrained model on a pair with exact flow *(core concept: EPE and the zero-flow baseline)* → 5 recurrent updates and degenerate probes *(evaluation practice)* → 6 the labelled dataset and validation → 7 split and pre-adaptation baseline *(evaluation practice)* → 8 bounded fine-tuning *(core concept)* → 9 held-out evaluation → 10 unseen pairs → 11 adapter export and reload *(engineering)* → 12 outputs → 13 optional BYOD → interpretation, troubleshooting, glossary and your conclusion.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). A CUDA GPU such as a Colab or Kaggle T4 is the documented runtime for the fine-tuning stages and is used automatically when present; the notebook also runs on CPU, much more slowly. Measured with the defaults: on the recorded Kaggle Tesla T4 run of 25 September 2026 the stages after setup took about 35 s (the fine-tune cell 8.9 s); a local CPU run of the same stages took about 139 s (the fine-tune about 96 s; review probe, 4 October 2026). Building the isolated environment the first time (the pinned `torch==2.14.0` wheel is its largest download; reused on a re-run) is the slowest setup step and adds a few minutes.
- **Knowledge:** basic Python, PIL and NumPy; images as `(H, W)` pixel grids; a flow field as an `(H, W, 2)` array of `(u, v)` displacements in pixels; and the Euclidean distance between two vectors.
- **Data:** the default path generates everything in code with `samples.py` and downloads no dataset: one 256×256 demonstration pair and a labelled dataset (40 pairs by default, `N_PAIRS`), each with exact flow and a validity mask. BYOD is optional and off by default. Expected BYOD input: two frames of one size, or a directory holding `pairs.json` — a list of `{'frame1': 'a.png', 'frame2': 'b.png', 'flow': 'a.flo', 'valid': 'a_valid.png'}` objects, where `flow` is a Middlebury `.flo` file or a `.npy` array of shape `(H, W, 2)` and `valid` is optional — and the files it names. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so; uploaded inputs stay in this runtime and are not sent to any inference API.
- **External access:** `download.pytorch.org` only, to fetch the pinned `torchvision/raft_large` checkpoint (~21 MB) at revision `ff5fadd56d26…`. No GitHub access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 33 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab and Kaggle import some packages, NumPy and often PyTorch among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env_<lock digest>/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 33 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins in the cell) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message.

The same cell then starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. It is the only cell that runs in the kernel (it is marked `# dimer: kernel cell`), and it is safe to re-run: a complete environment built from this exact lock is reused rather than rebuilt, and a live worker is kept with every variable created so far, so the cells after it keep working. To start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated environment and route later cells to it
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from multiprocessing.connection import Connection
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = '72a13d82ae25ed1600dad435ffce81e7debca26d1a6f8837aacb17942d4fac01'
LOCKED_PACKAGES = 33
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.12 \
    --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8 \
    --hash=sha256:cf42711a7ac791818b299fab0332a088c65aeeefa36290de98db92c434303b0c
    # via torch
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via torch
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   raft-optical-flow-pipeline (pyproject.toml)
    #   torchvision
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   raft-optical-flow-pipeline (pyproject.toml)
    #   torchvision
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via raft-optical-flow-pipeline (pyproject.toml)
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   raft-optical-flow-pipeline (pyproject.toml)
    #   torchvision
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via raft-optical-flow-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via torch
'''

SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
# One environment per lock digest: a re-run of this cell, or a second Run all in the same runtime, reuses a complete
# environment built from this exact lock instead of rebuilding it; a different lock gets a different folder.
ISOLATED_ENV = Path(os.environ.get('DIMER_ISOLATED_ENV', 'dimer_isolated_env_' + LOCK_SHA256[:12])).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / 'bin' / 'python'
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + '_tools')
ISOLATED_READY = ISOLATED_ENV / '.dimer-lock-sha256'


def _isolated_environment_ready():
    return ISOLATED_PYTHON.is_file() and ISOLATED_READY.is_file() and ISOLATED_READY.read_text(encoding='utf-8').strip() == LOCK_SHA256


if SKIP_INSTALL:
    print('DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.')
elif _isolated_environment_ready():
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': True, 'lock_sha256': LOCK_SHA256[:16] + '...', 'locked_packages': LOCKED_PACKAGES, 'kernel_python': platform.python_version()})
else:
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.')
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode('utf-8')).hexdigest() != LOCK_SHA256:
        raise RuntimeError('The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.')
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding='utf-8', newline='\n')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith('.data/scripts/uv'))
        uv = ISOLATED_TOOLS / 'uv'
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ, MPLBACKEND='Agg')
    for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP'):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), 'venv', '--quiet', '--managed-python', '--python', MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), '-c', 'import platform; print(platform.python_version())'], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f'{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.')
    ISOLATED_READY.unlink(missing_ok=True)
    subprocess.run([str(uv), 'pip', 'install', '--quiet', '--python', str(ISOLATED_PYTHON), '--require-hashes', '--only-binary', ':all:', '--index-url', 'https://pypi.org/simple', '-r', str(lock_path)], env=uv_env, check=True)
    ISOLATED_READY.write_text(LOCK_SHA256 + '\n', encoding='utf-8')
    print({'isolated_environment': str(ISOLATED_ENV), 'reused': False, 'isolated_python': isolated_version, 'kernel_python': platform.python_version(), 'locked_packages': LOCKED_PACKAGES, 'setup_seconds': round(time.perf_counter() - setup_started)})

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        self.python = str(python)
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def alive(self):
        return self.proc.poll() is None

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]

if SKIP_INSTALL:
    print('Routing disabled: the notebook runs in this kernel.')
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup if getattr(t, '__name__', '') != '_route_to_isolated_runtime']
    # Idempotent: a live worker on this environment is kept, with every variable the later cells created, so
    # re-running this cell alone never strands the cells after it. A dead or different worker is replaced.
    _previous = globals().get('_DIMER_ISOLATED_RUNTIME')
    _reuse = _previous is not None and getattr(_previous, 'python', None) == str(ISOLATED_PYTHON) and _previous.proc.poll() is None
    if _reuse:
        _DIMER_ISOLATED_RUNTIME = _previous
    else:
        if _previous is not None and _previous.proc.poll() is None:
            _previous.close()
        _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print({'routed_to': str(ISOLATED_PYTHON), 'worker_pid': _DIMER_ISOLATED_RUNTIME.proc.pid, 'worker_reused': _reuse})

### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `torchvision`, `numpy`, `PIL` versions, and whether CUDA is available.

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import os
import platform
import sys

NOTEBOOK_SOURCE = {
    'repository': 'raft-optical-flow-pipeline',
    'repository_revision': '1d4a15b255ca2dfb76ad544c6c312cf87f96eb67',
    'embedded_module': 'src/raft_optical_flow_pipeline/pipeline.py',
    'embedded_modules': ['src/raft_optical_flow_pipeline/samples.py', 'src/raft_optical_flow_pipeline/pipeline.py'],
    'module_sha256': 'ea1f6e775e44a1744952e0276e8a7295523a0456be63b5d27fbe553dbdb1a45a',
    'generator': 'build_notebook.py/2.2',
    'notebook_spec': '2.2',
}
import torch, torchvision, numpy, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'executable': sys.executable, 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'numpy': numpy.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/raft_optical_flow_pipeline/` @ `1d4a15b255ca`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/raft_optical_flow_pipeline/samples.py`

In [ ]:
"""Deterministic in-code frame pairs with exact optical flow, and Middlebury ``.flo`` input/output.

Nothing here is downloaded and nothing needs torch — numpy and Pillow only. Each pair is rendered from a
larger textured canvas:

* the **background** is a window into the canvas; frame 2's window is shifted by a whole number of pixels,
  so every background pixel moves by the same known displacement;
* each **object** is a textured disc or square pasted at one position in frame 1 and at a displaced
  position in frame 2, so its pixels move by that object's own known displacement.

The flow is defined from frame 1 to frame 2: the pixel at ``(x, y)`` in frame 1 appears at
``(x + u, y + v)`` in frame 2. A pixel is **valid** when its correspondence is visible in frame 2: object
pixels always are (objects stay inside the frame and do not overlap), background pixels are not when an
object covers their destination or when the destination leaves the frame. Scores are computed on valid
pixels only.
"""

from __future__ import annotations

from typing import Any

import numpy as np
from PIL import Image

PAIR_SIZE = (256, 256)  # (width, height), a multiple of 8 as RAFT requires
MAX_OBJECTS = 4
FLO_MAGIC = 202021.25  # Middlebury .flo header tag


def _texture(rng: np.random.Generator, height: int, width: int) -> np.ndarray:
    """Band-limited colour noise: enough structure for matching at every scale RAFT looks at."""
    base = rng.integers(0, 256, (height // 8 + 2, width // 8 + 2, 3), dtype=np.uint8)
    coarse = np.asarray(Image.fromarray(base).resize((width, height), Image.BICUBIC), dtype=np.float32)
    fine = rng.integers(0, 64, (height, width, 3)).astype(np.float32)
    return np.clip(0.8 * coarse + fine, 0, 255).astype(np.uint8)


def _shape_mask(kind: str, radius: int) -> np.ndarray:
    side = 2 * radius + 1
    yy, xx = np.mgrid[:side, :side]
    if kind == "disc":
        return (xx - radius) ** 2 + (yy - radius) ** 2 <= radius**2
    return np.ones((side, side), dtype=bool)


def moving_shapes_pair(
    seed: int = 0,
    *,
    size: tuple[int, int] = PAIR_SIZE,
    max_background_shift: int = 6,
    max_object_shift: int = 12,
    n_objects: int | None = None,
) -> dict[str, Any]:
    """One frame pair with its exact flow ``(H, W, 2)`` float32 (u, v) and boolean valid mask ``(H, W)``."""
    width, height = size
    if width % 8 or height % 8 or min(width, height) < 128:
        raise ValueError(f"size must be multiples of 8 and at least 128 px, got {size}")
    rng = np.random.default_rng(seed)
    margin = max_background_shift
    canvas = _texture(rng, height + 2 * margin, width + 2 * margin)
    bx, by = (int(v) for v in rng.integers(-max_background_shift, max_background_shift + 1, 2))
    frame1 = canvas[margin : margin + height, margin : margin + width].copy()
    frame2 = canvas[margin + by : margin + by + height, margin + bx : margin + bx + width].copy()
    # A window shifted by (bx, by) shows the content moved by (-bx, -by).
    flow = np.zeros((height, width, 2), dtype=np.float32)
    flow[..., 0], flow[..., 1] = -bx, -by
    ys, xs = np.mgrid[:height, :width]
    valid = (xs - bx >= 0) & (xs - bx < width) & (ys - by >= 0) & (ys - by < height)

    count = int(rng.integers(1, MAX_OBJECTS + 1)) if n_objects is None else n_objects
    placed: list[tuple[int, int, int]] = []
    covered2 = np.zeros((height, width), dtype=bool)
    objects = []
    attempts = 0
    while len(objects) < count and attempts < 200:
        attempts += 1
        radius = int(rng.integers(14, 34))
        dx, dy = (int(v) for v in rng.integers(-max_object_shift, max_object_shift + 1, 2))
        lo = radius + max_object_shift + 1
        cx, cy = int(rng.integers(lo, width - lo)), int(rng.integers(lo, height - lo))
        reach = radius + max_object_shift + 2
        if any(abs(cx - px) < reach + pr and abs(cy - py) < reach + pr for px, py, pr in placed):
            continue
        placed.append((cx, cy, radius))
        kind = "disc" if rng.random() < 0.5 else "square"
        mask = _shape_mask(kind, radius)
        patch = _texture(rng, mask.shape[0], mask.shape[1])
        y0, x0 = cy - radius, cx - radius
        region1 = (slice(y0, y0 + mask.shape[0]), slice(x0, x0 + mask.shape[1]))
        region2 = (slice(y0 + dy, y0 + dy + mask.shape[0]), slice(x0 + dx, x0 + dx + mask.shape[1]))
        frame1[region1][mask] = patch[mask]
        frame2[region2][mask] = patch[mask]
        flow[region1][mask] = (dx, dy)
        valid[region1] |= mask
        covered2[region2] |= mask
        objects.append({"kind": kind, "center": [cx, cy], "radius": radius, "shift": [dx, dy]})

    # A background pixel whose destination is covered by an object in frame 2 is occluded.
    object_pixels = np.zeros((height, width), dtype=bool)
    for obj in objects:
        cx, cy, r = obj["center"][0], obj["center"][1], obj["radius"]
        m = _shape_mask(obj["kind"], r)
        object_pixels[cy - r : cy + r + 1, cx - r : cx + r + 1] |= m
    dest_x = np.clip(xs - bx, 0, width - 1)
    dest_y = np.clip(ys - by, 0, height - 1)
    occluded = ~object_pixels & covered2[dest_y, dest_x]
    valid &= ~occluded
    return {
        "id": f"pair-{seed}",
        "image1": Image.fromarray(frame1),
        "image2": Image.fromarray(frame2),
        "flow": flow,
        "valid": valid,
        "background_shift": [-bx, -by],
        "objects": objects,
    }


def flow_dataset(
    n_pairs: int = 40, *, seed: int = 0, size: tuple[int, int] = PAIR_SIZE
) -> list[dict[str, Any]]:
    """A deterministic list of ``moving_shapes_pair`` records, the adaptation dataset of the tutorial."""
    if not 1 <= n_pairs <= 1000:
        raise ValueError(f"n_pairs must be in 1..1000, got {n_pairs}")
    return [moving_shapes_pair(seed * 100_000 + i, size=size) for i in range(n_pairs)]


def split_pairs(
    records: list[dict[str, Any]], *, train_fraction: float = 0.75, seed: int = 0
) -> tuple[list, list]:
    """Seeded random split at the pair level; pairs cut from the same video must be split by video instead."""
    if not 0.0 < train_fraction < 1.0:
        raise ValueError(f"train_fraction must be between 0 and 1, got {train_fraction}")
    if len(records) < 2:
        raise ValueError(f"at least 2 records are required to split, got {len(records)}")
    order = np.random.default_rng(seed).permutation(len(records))
    n_train = min(len(records) - 1, max(1, int(len(records) * train_fraction)))
    chosen = {int(i) for i in order[:n_train]}
    return [r for i, r in enumerate(records) if i in chosen], [
        r for i, r in enumerate(records) if i not in chosen
    ]


def static_pair(size: tuple[int, int] = PAIR_SIZE, seed: int = 7) -> dict[str, Any]:
    """The same textured frame twice: the true flow is zero everywhere."""
    width, height = size
    frame = Image.fromarray(_texture(np.random.default_rng(seed), height, width))
    return {
        "id": "static",
        "image1": frame,
        "image2": frame.copy(),
        "flow": np.zeros((height, width, 2), np.float32),
        "valid": np.ones((height, width), bool),
    }


def blank_pair(size: tuple[int, int] = PAIR_SIZE) -> dict[str, Any]:
    """Two featureless white frames: any motion is undetermined, and a flow estimate is a guess."""
    width, height = size
    frame = Image.new("RGB", size, (255, 255, 255))
    return {
        "id": "blank",
        "image1": frame,
        "image2": frame.copy(),
        "flow": np.zeros((height, width, 2), np.float32),
        "valid": np.ones((height, width), bool),
    }


def read_flo(path: Any) -> np.ndarray:
    """Read a Middlebury ``.flo`` file into an ``(H, W, 2)`` float32 array; refuse a bad header or size."""
    with open(path, "rb") as fh:
        magic = np.fromfile(fh, np.float32, count=1)
        if magic.size != 1 or magic[0] != FLO_MAGIC:
            raise ValueError(f"{path}: not a Middlebury .flo file (bad magic)")
        width, height = (int(v) for v in np.fromfile(fh, np.int32, count=2))
        if not (0 < width <= 4096 and 0 < height <= 4096):
            raise ValueError(f"{path}: implausible .flo size {width}x{height}")
        data = np.fromfile(fh, np.float32, count=2 * width * height)
    if data.size != 2 * width * height:
        raise ValueError(f"{path}: truncated .flo data")
    return data.reshape(height, width, 2)


def write_flo(path: Any, flow: np.ndarray) -> None:
    """Write an ``(H, W, 2)`` flow array as a Middlebury ``.flo`` file."""
    flow = np.asarray(flow, dtype=np.float32)
    if flow.ndim != 3 or flow.shape[2] != 2:
        raise ValueError(f"flow must have shape (H, W, 2), got {flow.shape}")
    with open(path, "wb") as fh:
        np.array([FLO_MAGIC], np.float32).tofile(fh)
        np.array([flow.shape[1], flow.shape[0]], np.int32).tofile(fh)
        flow.tofile(fh)

**Module 2/2:** `src/raft_optical_flow_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Dense optical flow and bounded fine-tuning with torchvision's RAFT-Large (``C_T_SKHT_V2`` weights).

The model is torchvision's own ``raft_large`` architecture, built with ``weights=None`` so that nothing is
downloaded at construction, then loaded from one digest-verified local checkpoint file (``weights/<key>/``)
with ``torch.load(..., weights_only=True)``, which refuses arbitrary pickled objects.

Until ``tools/pin_snapshot.py`` has recorded the checkpoint's SHA-256 and byte size, the package refuses to
stage, verify or load weights: an unpinned checkpoint is never trusted.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .samples import read_flo` removed — names are kernel globals defined by the carried modules

MODEL_ID = "torchvision/raft_large"
# The pinned identity of a URL-hosted checkpoint is the SHA-256 of its bytes. torchvision names the file
# after the first 8 hex digits of that digest, so the URL is content-addressed; the full digest is pinned.
MODEL_REVISION = "ff5fadd56d26b40647388883af1547351ea17868b765c05b27231e72dd16a322"
MODEL_LICENSE = "bsd-3-clause"
MODEL_KEY = "raft-large-c-t-skht-v2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_FORMAT = "raft-adapter-v1"
UNPINNED = "unpinned"
PIN_COMMAND = "python tools/pin_snapshot.py"
WEIGHTS_FILE = "raft_large_C_T_SKHT_V2-ff5fadd5.pth"
WEIGHTS_URL = f"https://download.pytorch.org/models/{WEIGHTS_FILE}"
WEIGHTS_SHA256_PREFIX = "ff5fadd5"
TORCHVISION_WEIGHTS = "Raft_Large_Weights.C_T_SKHT_V2"

# Operational ceilings. RAFT's all-pairs correlation volume holds (H/8 * W/8)^2 values, so memory grows with
# the square of the pixel count: 1024 x 768 already needs about 0.6 GB for it alone.
MIN_IMAGE_SIDE = 128  # torchvision's weight metadata: min_size (128, 128)
MAX_IMAGE_SIDE = 1024
MAX_PIXELS = 1024 * 768
MAX_RECORDS = 2000
DEFAULT_FLOW_UPDATES = 12  # torchvision's default num_flow_updates
MAX_FLOW = 400.0  # the reference sequence loss ignores ground-truth displacements at or above this (pixels)

# Bounded fine-tuning defaults, following torchvision's optical-flow reference (references/optical_flow):
# AdamW, sequence loss with gamma 0.8, gradient-norm clipping at 1.0, BatchNorm frozen as in its Sintel stage.
DEFAULT_EPOCHS = 3
DEFAULT_BATCH_SIZE = 2
DEFAULT_LEARNING_RATE = 2e-5
DEFAULT_WEIGHT_DECAY = 5e-5
DEFAULT_GAMMA = 0.8
DEFAULT_TRAIN_UPDATES = 12
DEFAULT_SEED = 20260925
ENCODER_PREFIXES = ("feature_encoder.", "context_encoder.")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def is_pinned() -> bool:
    """True once MODEL_REVISION is the checkpoint's 64-hex SHA-256."""
    revision = MODEL_REVISION
    return len(revision) == 64 and all(c in "0123456789abcdef" for c in revision)


def _require_pinned(action: str) -> None:
    if not is_pinned():
        raise RuntimeError(
            f"refusing to {action}: {MODEL_ID} has no pinned checkpoint digest yet (MODEL_REVISION = "
            f"{MODEL_REVISION!r}); run `{PIN_COMMAND}` to record the SHA-256 and byte size"
        )


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local checkpoint against its DIMER manifest; raise naming the first mismatch."""
    _require_pinned("verify the checkpoint")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        if not entry.get("sha256") or entry.get("bytes") is None:
            raise ValueError(f"{entry['path']}: manifest records no sha256/bytes; run `{PIN_COMMAND}`")
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"checkpoint file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _url_download(entry: Mapping[str, Any], root: Path) -> None:
    """Fetch one manifest entry from its URL; torch.hub checks the filename's SHA-256 prefix on the way in."""
    from torch.hub import download_url_to_file

    download_url_to_file(
        entry["url"], str(root / entry["path"]), hash_prefix=WEIGHTS_SHA256_PREFIX, progress=False
    )


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[Mapping[str, Any], Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    _require_pinned("stage the checkpoint")
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"checkpoint at {root} is missing {[e['path'] for e in missing]}; "
            "pass allow_download=True to fetch it"
        )
    fetch = downloader or _url_download
    for entry in missing:
        if entry.get("url") != WEIGHTS_URL:
            raise ValueError(f"manifest URL {entry.get('url')!r} != package URL {WEIGHTS_URL!r}")
        fetch(entry, root)
    return [entry["path"] for entry in missing]


# --------------------------------------------------------------------------- metrics


def flow_metrics(predicted: np.ndarray, truth: np.ndarray, valid: np.ndarray | None = None) -> dict[str, Any]:
    """End-point error (EPE), outlier rates and angular error of a flow field against ground truth.

    ``EPE`` is the mean Euclidean distance in pixels between predicted and true displacement over valid
    pixels. ``1px``/``3px``/``5px`` are the fractions of valid pixels whose error is below that many pixels
    (higher is better), as torchvision's reference evaluation reports them. ``angular_error_deg`` is the mean
    angle between the space-time vectors (u, v, 1), the Middlebury convention.
    """
    predicted = np.asarray(predicted, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    if predicted.shape != truth.shape or truth.ndim != 3 or truth.shape[2] != 2:
        raise ValueError(f"flows must both have shape (H, W, 2), got {predicted.shape} and {truth.shape}")
    mask = np.ones(truth.shape[:2], dtype=bool) if valid is None else np.asarray(valid, dtype=bool)
    if mask.shape != truth.shape[:2]:
        raise ValueError(f"valid mask shape {mask.shape} != flow shape {truth.shape[:2]}")
    mask = mask & (np.linalg.norm(truth, axis=2) < MAX_FLOW)
    if not mask.any():
        raise ValueError("no valid pixel to score")
    error = np.linalg.norm(predicted - truth, axis=2)[mask]
    p, t = predicted[mask], truth[mask]
    # Angle between (u, v, 1) vectors as atan2(|a x b|, a . b): exact near 0, where arccos is not.
    a = np.column_stack([p, np.ones(len(p))])
    b = np.column_stack([t, np.ones(len(t))])
    angle = np.arctan2(np.linalg.norm(np.cross(a, b), axis=1), (a * b).sum(axis=1))
    return {
        "epe": float(error.mean()),
        "1px": float((error < 1).mean()),
        "3px": float((error < 3).mean()),
        "5px": float((error < 5).mean()),
        "angular_error_deg": float(np.degrees(angle).mean()),
        "valid_pixels": int(mask.sum()),
    }


def aggregate_metrics(rows: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Mean of per-pair metrics (each pair weighs the same) plus the pixel-weighted EPE."""
    if not rows:
        raise ValueError("no rows to aggregate")
    keys = ("epe", "1px", "3px", "5px", "angular_error_deg")
    out: dict[str, Any] = {key: float(np.mean([row[key] for row in rows])) for key in keys}
    pixels = sum(row["valid_pixels"] for row in rows)
    out["pixel_weighted_epe"] = float(sum(row["epe"] * row["valid_pixels"] for row in rows) / pixels)
    out["n_pairs"] = len(rows)
    return out


# --------------------------------------------------------------------------- input validation


def validate_pair(image1: Any, image2: Any) -> tuple[Image.Image, Image.Image]:
    """Two PIL images of one size inside the ceilings; returns them converted to RGB."""
    for name, image in (("image1", image1), ("image2", image2)):
        if not isinstance(image, Image.Image):
            raise TypeError(f"{name} must be a PIL.Image.Image, got {type(image).__name__}")
    if image1.size != image2.size:
        raise ValueError(f"frames differ in size: {image1.size} vs {image2.size}")
    width, height = image1.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"frame side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"frame side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    if width * height > MAX_PIXELS:
        raise ValueError(f"frame has {width * height} pixels > MAX_PIXELS {MAX_PIXELS}")
    return image1.convert("RGB"), image2.convert("RGB")


def _check_updates(value: Any) -> int:
    if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= 32:
        raise ValueError(f"num_flow_updates must be an int in 1..32, got {value!r}")
    return value


INPUT_SCHEMA: dict[str, Any] = {
    "input": "two PIL.Image.Image frames of the same size (any mode, converted to RGB); frame 1 then frame 2",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "max_pixels": MAX_PIXELS,
    "num_flow_updates": [1, 32],
    "output": "flow (H, W, 2) float32 in pixels: frame-1 pixel (x, y) moves to (x + u, y + v) in frame 2",
    "preprocessing": (
        "RGB to float in [0, 1], normalised to [-1, 1] (torchvision's OpticalFlow preset); frames are padded "
        "by edge replication to a multiple of 8 px and the flow is cropped back"
    ),
}


def validate_inputs(
    image1: Any, image2: Any, *, num_flow_updates: int = DEFAULT_FLOW_UPDATES
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict)."""
    first, second = validate_pair(image1, image2)
    updates = _check_updates(num_flow_updates)
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": name, "mode": img.mode, "size": list(img.size)}
            for name, img in (("frame1", image1), ("frame2", image2))
        ],
        "num_flow_updates": updates,
        "padding": [(-first.width) % 8, (-first.height) % 8],
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def validate_dataset(records: Sequence[Mapping[str, Any]], *, epochs: int = DEFAULT_EPOCHS) -> dict[str, Any]:
    """Validation stage for flow records ``{"image1", "image2", "flow", "valid"?}``: raise on the first broken
    record, else return the dataset manifest with displacement statistics."""
    if not records:
        raise ValueError("dataset must hold at least one record")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset holds {len(records)} records > MAX_RECORDS {MAX_RECORDS}")
    if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
        raise ValueError(f"epochs must be an int in 1..100, got {epochs!r}")
    magnitudes, valid_fraction, sizes = [], [], set()
    findings = []
    for idx, record in enumerate(records):
        if not isinstance(record, Mapping) or not {"image1", "image2", "flow"} <= set(record):
            raise ValueError(f"record {idx} must be a mapping with 'image1', 'image2' and 'flow'")
        try:
            first, _second = validate_pair(record["image1"], record["image2"])
        except (TypeError, ValueError) as exc:
            raise type(exc)(f"record {idx}: {exc}") from exc
        flow = np.asarray(record["flow"])
        if flow.shape != (first.height, first.width, 2):
            raise ValueError(f"record {idx}: flow shape {flow.shape} != ({first.height}, {first.width}, 2)")
        if not np.isfinite(flow).all():
            raise ValueError(f"record {idx}: flow contains non-finite values")
        valid = np.asarray(record.get("valid", np.ones(flow.shape[:2], bool)))
        if valid.shape != flow.shape[:2] or (valid.dtype != bool and not np.isin(valid, (0, 1)).all()):
            raise ValueError(f"record {idx}: valid mask must be boolean with shape {flow.shape[:2]}")
        valid = valid.astype(bool)
        if not valid.any():
            raise ValueError(f"record {idx}: valid mask selects no pixel")
        magnitude = np.linalg.norm(flow, axis=2)[valid]
        if (magnitude >= MAX_FLOW).any():
            findings.append(
                f"record {idx}: {int((magnitude >= MAX_FLOW).sum())} valid pixels move >= {MAX_FLOW} px "
                "and are ignored by the loss"
            )
        magnitudes.append(float(magnitude.mean()))
        valid_fraction.append(float(valid.mean()))
        sizes.add(first.size)
    if len(sizes) > 1:
        findings.append(f"{len(sizes)} different frame sizes; batches are grouped by size")
    return {
        "n_records": len(records),
        "frame_sizes": sorted([list(s) for s in sizes]),
        "mean_displacement_px": float(np.mean(magnitudes)),
        "max_mean_displacement_px": float(np.max(magnitudes)),
        "mean_valid_fraction": float(np.mean(valid_fraction)),
        "epochs": epochs,
        "findings": findings,
        "verdict": "accepted",
    }


def read_flow_records(directory: str | Path) -> list[dict[str, Any]]:
    """Read BYOD flow records from ``<directory>/pairs.json`` plus the files it names.

    ``pairs.json`` is a list of objects such as
    ``{"frame1": "a.png", "frame2": "b.png", "flow": "a.flo", "valid": "a_valid.png"}``; ``flow`` is a
    Middlebury ``.flo`` file or a ``.npy`` array of shape (H, W, 2), and ``valid`` is optional (a
    single-channel PNG whose non-zero pixels are valid). File names must stay inside ``directory``.
    """
    root = Path(directory).resolve()
    index = root / "pairs.json"
    if not index.is_file():
        raise FileNotFoundError(f"{index} not found; expected pairs.json next to the frames")
    entries = json.loads(index.read_text(encoding="utf-8"))
    if not isinstance(entries, list) or not entries:
        raise ValueError("pairs.json must hold a non-empty list of records")
    if len(entries) > MAX_RECORDS:
        raise ValueError(f"pairs.json lists {len(entries)} records > MAX_RECORDS {MAX_RECORDS}")

    def resolve(idx: int, name: Any) -> Path:
        relative = Path(str(name))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"pairs.json entry {idx}: file {name!r} must be relative to {root}")
        resolved = (root / relative).resolve()
        if root not in resolved.parents:
            raise ValueError(f"pairs.json entry {idx}: file {name!r} resolves outside {root}")
        return resolved

    records = []
    for idx, entry in enumerate(entries):
        if not isinstance(entry, dict) or not {"frame1", "frame2", "flow"} <= set(entry):
            raise ValueError(f"pairs.json entry {idx} must have 'frame1', 'frame2' and 'flow'")
        paths = {key: resolve(idx, entry[key]) for key in ("frame1", "frame2", "flow")}
        frames = []
        for key in ("frame1", "frame2"):
            with Image.open(paths[key]) as handle:
                frames.append(handle.convert("RGB"))
        flow = (
            np.load(paths["flow"], allow_pickle=False)
            if paths["flow"].suffix == ".npy"
            else read_flo(paths["flow"])
        )
        record = {
            "id": str(entry["frame1"]),
            "image1": frames[0],
            "image2": frames[1],
            "flow": flow.astype(np.float32),
        }
        if entry.get("valid"):
            with Image.open(resolve(idx, entry["valid"])) as handle:
                record["valid"] = np.asarray(handle.convert("L")) > 0
        records.append(record)
    return records


def evaluation_report(
    result: Mapping[str, Any],
    truth: np.ndarray | None = None,
    valid: np.ndarray | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Single-pair evaluation stage: EPE, outlier rates and angular error against ground truth, next to
    the zero-flow baseline (predicting no motion), or ``not-measurable`` without ground truth."""
    flow = np.asarray(result["flow"])
    base = {
        "task": "dense optical flow between two frames",
        "sample_kind": sample_kind,
        "flow_shape": list(flow.shape),
        "num_flow_updates": result.get("num_flow_updates", DEFAULT_FLOW_UPDATES),
        "mean_predicted_displacement_px": float(np.linalg.norm(flow, axis=2).mean()),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if truth is None:
        return {
            **base,
            "metrics": {},
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth flow was supplied for the evaluated pair",
            "needs": (
                "frame pairs from your own footage with ground-truth flow (rendered or measured), "
                "scored by EPE and outlier rates against the zero-flow baseline"
            ),
        }
    metrics = flow_metrics(flow, truth, valid)
    baseline = flow_metrics(np.zeros_like(np.asarray(truth, dtype=np.float32)), truth, valid)
    return {
        **base,
        "metrics": metrics,
        "baselines": [{"id": "zero-flow", **baseline}],
        "verdict": "sample-sanity",
        "reason": "one frame pair with exact ground truth; geometry sanity evidence, not a benchmark",
        "needs": "a labelled set of frame pairs from the deployment domain for any EPE claim",
    }


# --------------------------------------------------------------------------- model


def build_model() -> Any:
    """torchvision's raft_large architecture with random weights and nothing downloaded."""
    from torchvision.models.optical_flow import raft_large

    return raft_large(weights=None, progress=False)


def _to_tensor(images: Sequence[Image.Image]) -> Any:
    """torchvision's OpticalFlow preset: uint8 RGB to float in [0, 1], then normalised to [-1, 1]."""
    import torch

    arrays = np.stack([np.asarray(image, dtype=np.float32) / 255.0 for image in images])
    return torch.from_numpy(arrays).permute(0, 3, 1, 2) * 2.0 - 1.0


def _pad8(batch: Any) -> tuple[Any, tuple[int, int]]:
    import torch.nn.functional as F

    height, width = batch.shape[-2:]
    pad_h, pad_w = (-height) % 8, (-width) % 8
    if pad_h or pad_w:
        batch = F.pad(batch, (0, pad_w, 0, pad_h), mode="replicate")
    return batch, (height, width)


def _hold_batchnorm(module: Any) -> None:
    """Keep every BatchNorm layer under ``module`` in eval mode (RAFT's context encoder carries BatchNorm)."""
    import torch

    for sub in module.modules():
        if isinstance(sub, torch.nn.modules.batchnorm._BatchNorm):
            sub.eval()


def sequence_loss(predictions: Sequence[Any], truth: Any, valid: Any, gamma: float = DEFAULT_GAMMA) -> Any:
    """RAFT's loss: L1 flow error over every refinement, weighted gamma**(N - i), valid and < MAX_FLOW px only
    (torchvision references/optical_flow utils.sequence_loss)."""
    import torch

    if not 0.0 < gamma < 1.0:
        raise ValueError(f"gamma must be in (0, 1), got {gamma}")
    norm = torch.sum(truth**2, dim=1).sqrt()
    mask = (valid & (norm < MAX_FLOW))[:, None, :, :]
    stacked = torch.stack(list(predictions))
    per_iteration = ((stacked - truth).abs() * mask).mean(dim=(1, 2, 3, 4))
    weights = gamma ** torch.arange(
        len(predictions) - 1, -1, -1, device=truth.device, dtype=per_iteration.dtype
    )
    return (per_iteration * weights).sum()


@dataclass
class RaftPipeline:
    """Dense optical flow, and bounded fine-tuning on a caller's flow pairs, over RAFT-Large."""

    model: Any
    device: str
    source: str = "snapshot"
    base_state_digest: str | None = None
    adapted: bool = False
    frozen_prefixes: tuple[str, ...] = field(default_factory=tuple)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> RaftPipeline:
        _require_pinned("load the model")
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no checkpoint manifest at {root}; stage {MODEL_ID} under weights/{MODEL_KEY} "
                "(allow_download=True fetches the manifest-listed file)"
            )
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)

        import torch

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = build_model()
        state = torch.load(root / WEIGHTS_FILE, map_location="cpu", weights_only=True)
        model.load_state_dict(state, strict=True)
        return cls(
            model=model.to(resolved_device).eval(),
            device=resolved_device,
            source=str(root),
            base_state_digest=MODEL_REVISION,
        )

    def _forward(
        self, first: Sequence[Image.Image], second: Sequence[Image.Image], updates: int
    ) -> list[Any]:
        batch1, (height, width) = _pad8(_to_tensor(first).to(self.device))
        batch2, _ = _pad8(_to_tensor(second).to(self.device))
        predictions = self.model(batch1, batch2, num_flow_updates=updates)
        return [flow[..., :height, :width] for flow in predictions]

    def estimate(
        self, image1: Image.Image, image2: Image.Image, *, num_flow_updates: int = DEFAULT_FLOW_UPDATES
    ) -> dict[str, Any]:
        """Estimate the flow from ``image1`` to ``image2``; returns ``flow`` as an (H, W, 2) float32 array."""
        import torch

        first, second = validate_pair(image1, image2)
        updates = _check_updates(num_flow_updates)
        was_training = self.model.training
        self.model.eval()
        with torch.inference_mode():
            flow = self._forward([first], [second], updates)[-1][0]
        if was_training:
            self.model.train()
        array = flow.permute(1, 2, 0).float().cpu().numpy()
        if array.shape != (first.height, first.width, 2) or not np.isfinite(array).all():
            raise RuntimeError(f"backend returned a malformed flow of shape {array.shape}")
        return {
            "flow": array,
            "num_flow_updates": updates,
            "width": first.width,
            "height": first.height,
            "adapted": self.adapted,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, num_flow_updates: int = DEFAULT_FLOW_UPDATES
    ) -> dict[str, Any]:
        """Mean EPE, outlier rates and angular error over labelled pairs, next to the zero-flow baseline."""
        rows, zero_rows = [], []
        for record in records:
            flow = self.estimate(record["image1"], record["image2"], num_flow_updates=num_flow_updates)[
                "flow"
            ]
            truth = np.asarray(record["flow"], dtype=np.float32)
            valid = record.get("valid")
            rows.append(flow_metrics(flow, truth, valid))
            zero_rows.append(flow_metrics(np.zeros_like(truth), truth, valid))
        return {
            **aggregate_metrics(rows),
            "zero_flow_baseline": aggregate_metrics(zero_rows),
            "num_flow_updates": num_flow_updates,
            "adapted": self.adapted,
            "estimation": (
                f"one pass over {len(records)} pairs; each pair weighs the same; "
                "no resampling, no dispersion estimate"
            ),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        weight_decay: float = DEFAULT_WEIGHT_DECAY,
        num_flow_updates: int = DEFAULT_TRAIN_UPDATES,
        gamma: float = DEFAULT_GAMMA,
        seed: int = DEFAULT_SEED,
        freeze_encoders: bool = True,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning with RAFT's sequence loss, mutating this pipeline.

        Every BatchNorm layer is held in eval mode, as torchvision's reference does in its Sintel stage
        (``--freeze-batch-norm``). With ``freeze_encoders`` the feature and context encoders also keep their
        weights and only the recurrent update block and the upsampling mask predictor train. Gradients are
        clipped to norm 1.0. Frames in a batch must share a size.
        """
        import torch

        validate_dataset(records, epochs=epochs)
        updates = _check_updates(num_flow_updates)
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if (
            isinstance(learning_rate, bool)
            or not isinstance(learning_rate, int | float)
            or not 0.0 < float(learning_rate) <= 1.0
        ):
            raise ValueError(f"learning_rate must be a number in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed)
        for name, parameter in self.model.named_parameters():
            parameter.requires_grad = not (freeze_encoders and name.startswith(ENCODER_PREFIXES))
        self.frozen_prefixes = ENCODER_PREFIXES if freeze_encoders else ()
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=float(learning_rate), weight_decay=float(weight_decay))

        by_size: dict[tuple[int, int], list[int]] = {}
        for i, record in enumerate(records):
            by_size.setdefault(record["image1"].size, []).append(i)
        epoch_losses: list[float] = []
        for epoch in range(epochs):
            self.model.train()
            _hold_batchnorm(self.model)
            batches = []
            for indices in by_size.values():
                order = [indices[int(j)] for j in rng.permutation(len(indices))]
                batches.extend(order[k : k + batch_size] for k in range(0, len(order), batch_size))
            running, n_batches = 0.0, 0
            for b in rng.permutation(len(batches)):
                batch = [records[i] for i in batches[int(b)]]
                first = [validate_pair(r["image1"], r["image2"])[0] for r in batch]
                second = [validate_pair(r["image1"], r["image2"])[1] for r in batch]
                truth = (
                    torch.from_numpy(np.stack([np.asarray(r["flow"], np.float32) for r in batch]))
                    .permute(0, 3, 1, 2)
                    .to(self.device)
                )
                valid = torch.from_numpy(
                    np.stack(
                        [np.asarray(r.get("valid", np.ones(r["flow"].shape[:2], bool)), bool) for r in batch]
                    )
                ).to(self.device)
                optimizer.zero_grad(set_to_none=True)
                loss = sequence_loss(self._forward(first, second, updates), truth, valid, gamma)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
                optimizer.step()
                running += float(loss.detach().cpu())
                n_batches += 1
            epoch_losses.append(running / max(1, n_batches))
            if progress is not None:
                progress({"epoch": epoch + 1, "epochs": epochs, "loss": epoch_losses[-1]})
        self.model.eval()
        self.adapted = True
        return {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "weight_decay": float(weight_decay),
            "optimizer": "AdamW",
            "loss": f"RAFT sequence loss (L1, gamma {gamma}, displacements >= {MAX_FLOW:.0f} px ignored)",
            "gradient_clip_norm": 1.0,
            "batchnorm": "frozen (eval mode)",
            "num_flow_updates": updates,
            "seed": seed,
            "precision": "float32",
            "freeze_encoders": freeze_encoders,
            "frozen_prefixes": list(self.frozen_prefixes),
            "trainable_parameters": sum(p.numel() for p in trainable),
            "total_parameters": sum(p.numel() for p in self.model.parameters()),
            "epoch_losses": epoch_losses,
            "final_loss": epoch_losses[-1] if epoch_losses else None,
            "device": self.device,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def save_artifact(self, path: str | Path, *, notes: str | None = None) -> dict[str, Any]:
        """Write the adapted tensors as one SafeTensors file with the provenance in its metadata.

        Tensors under ``frozen_prefixes`` are left out: they equal the verified base checkpoint, which
        ``load_artifact`` loads first. The artifact is therefore an adapter bound to the base checkpoint.
        """
        from safetensors.torch import save_file

        if not self.adapted:
            raise RuntimeError("nothing to export: the pipeline has not been fine-tuned")
        artifact_path = Path(path)
        artifact_path.parent.mkdir(parents=True, exist_ok=True)
        tensors = {
            name: value.detach().cpu().contiguous().clone()
            for name, value in self.model.state_dict().items()
            if not any(name.startswith(prefix) for prefix in self.frozen_prefixes)
        }
        metadata = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_key": MODEL_KEY,
            "frozen_prefixes": json.dumps(list(self.frozen_prefixes)),
            "base_state_digest": self.base_state_digest or "",
            "notes": notes or "",
        }
        save_file(tensors, str(artifact_path), metadata=metadata)
        return {
            "path": str(artifact_path),
            "bytes": artifact_path.stat().st_size,
            "sha256": _sha256(artifact_path),
            "format": ARTIFACT_FORMAT,
            "tensors": len(tensors),
            "frozen_prefixes": list(self.frozen_prefixes),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @staticmethod
    def read_artifact_metadata(path: str | Path) -> dict[str, Any]:
        """Read and check the artifact's provenance header without loading any tensor."""
        from safetensors import safe_open

        with safe_open(str(path), framework="pt") as handle:
            metadata = dict(handle.metadata() or {})
        if metadata.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {metadata.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if (metadata.get("model_id"), metadata.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(
                f"artifact was built on {metadata.get('model_id')}@{metadata.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        if metadata.get("model_key") != MODEL_KEY:
            raise ValueError(
                f"artifact was built on {metadata.get('model_key')!r}, package pins {MODEL_KEY!r}"
            )
        return {**metadata, "frozen_prefixes": json.loads(metadata["frozen_prefixes"])}

    def apply_artifact(self, path: str | Path) -> None:
        """Load adapter tensors onto this base pipeline; refuse any tensor set that does not fit."""
        from safetensors.torch import load_file

        metadata = self.read_artifact_metadata(path)
        expected_base = metadata.get("base_state_digest") or None
        if expected_base and self.base_state_digest and expected_base != self.base_state_digest:
            raise ValueError("artifact was exported against a different base checkpoint digest")
        tensors = load_file(str(path), device="cpu")
        prefixes = tuple(metadata["frozen_prefixes"])
        result = self.model.load_state_dict(tensors, strict=False)
        if result.unexpected_keys:
            raise ValueError(
                f"artifact carries tensors the model does not have: {result.unexpected_keys[:5]}"
            )
        stray = [key for key in result.missing_keys if not any(key.startswith(p) for p in prefixes)]
        if stray:
            raise ValueError(f"artifact is missing trainable tensors: {stray[:5]}")
        self.model.eval()
        self.adapted = True
        self.frozen_prefixes = prefixes
        self.source = f"artifact:{Path(path).name}"

    @classmethod
    def load_artifact(
        cls, path: str | Path, *, weights_dir: str | Path | None = None, device: str | None = None
    ) -> RaftPipeline:
        """Rebuild an adapted pipeline: verified base checkpoint first, then the adapter tensors."""
        cls.read_artifact_metadata(path)
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir)
        pipe.apply_artifact(path)
        return pipe


def flow_to_rgb(flow: np.ndarray) -> Image.Image:
    """Colour-code a flow field (hue = direction, saturation = speed) with torchvision's flow_to_image."""
    import torch
    from torchvision.utils import flow_to_image

    tensor = torch.from_numpy(np.ascontiguousarray(np.asarray(flow, dtype=np.float32).transpose(2, 0, 1)))
    return Image.fromarray(flow_to_image(tensor).permute(1, 2, 0).numpy())

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the manifest URL on `download.pytorch.org` **and must hash to SHA-256 `ff5fadd56d26…`**, `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned snapshot, then load the model
import json

MANIFEST = {
  "format": "dimer_url_snapshot",
  "formatVersion": 1,
  "modelKey": "raft-large-c-t-skht-v2",
  "modelId": "torchvision/raft_large",
  "revision": "ff5fadd56d26b40647388883af1547351ea17868b765c05b27231e72dd16a322",
  "files": [
    {
      "path": "raft_large_C_T_SKHT_V2-ff5fadd5.pth",
      "url": "https://download.pytorch.org/models/raft_large_C_T_SKHT_V2-ff5fadd5.pth",
      "bytes": 21106607,
      "sha256": "ff5fadd56d26b40647388883af1547351ea17868b765c05b27231e72dd16a322"
    }
  ],
  "totalBytes": 21106607
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. What the pretrained model does on a pair with exact flow

Before adapting anything, inspect the model you start from. The carried `samples` module renders a deterministic frame pair from a textured canvas. The **background** moves by one whole-pixel displacement, and each textured disc or square **object** moves by its own. Because the pair is rendered, the true flow of every pixel is known exactly. A pixel is **valid** when its destination is visible in frame 2; background pixels that an object covers in frame 2, or that leave the frame, are not.

The metrics are the ones optical-flow benchmarks report, computed over valid pixels only:
- **EPE** (end-point error): the mean distance in pixels between the predicted and the true displacement. Lower is better.
- **1px / 3px / 5px**: the fraction of valid pixels whose error is below that many pixels. Higher is better.
- **Angular error**: the mean angle in degrees between the space-time vectors `(u, v, 1)` of the prediction and the truth.

Every score is printed next to the **zero-flow baseline**, which predicts no motion anywhere. Its EPE equals the mean true displacement, so a model is only useful where it beats that row. One pair gives the verdict `sample-sanity`, not a benchmark.

**Predict before running:** the background moves by about 7 px on average. What end-point error will zero flow score, and will RAFT be closer to 0.1 px or to 1 px?

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path

os.makedirs('outputs', exist_ok=True)
OUTPUTS = Path('outputs')

NUM_FLOW_UPDATES = 12  # @param {type:"integer"}
DEMO_SEED = 3  # @param {type:"integer"}

print({'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PIXELS': MAX_PIXELS,
       'DEFAULT_FLOW_UPDATES': DEFAULT_FLOW_UPDATES, 'MAX_FLOW': MAX_FLOW})

demo = moving_shapes_pair(DEMO_SEED)
buffer = io.BytesIO()
demo['image1'].save(buffer, format='PNG')
print({'sample_kind': 'synthetic', 'size': list(demo['image1'].size), 'sha256': hashlib.sha256(buffer.getvalue()).hexdigest()[:16],
       'background_shift': demo['background_shift'], 'objects': [(o['kind'], o['shift']) for o in demo['objects']],
       'valid_fraction': round(float(demo['valid'].mean()), 3)})

input_manifest = validate_inputs(demo['image1'], demo['image2'], num_flow_updates=NUM_FLOW_UPDATES)
try:
    validate_inputs(demo['image1'], demo['image2'].resize((128, 128)))
except ValueError as exc:
    input_manifest['findings'].append({'probe': 'frames of different sizes', 'rejected': str(exc)})
print({'verdict': input_manifest['verdict'], 'padding': input_manifest['padding'], 'findings': input_manifest['findings']})

demo_result = pipe.estimate(demo['image1'], demo['image2'], num_flow_updates=NUM_FLOW_UPDATES)
demo_report = evaluation_report(demo_result, demo['flow'], demo['valid'], sample_kind='synthetic')
zero_row = demo_report['baselines'][0]
print(f"{'':<10s} {'EPE':>8s} {'1px':>7s} {'3px':>7s} {'5px':>7s} {'angle':>8s}")
for name, row in (('RAFT', demo_report['metrics']), ('zero-flow', zero_row)):
    print(f"{name:<10s} {row['epe']:>8.3f} {row['1px']:>7.3f} {row['3px']:>7.3f} {row['5px']:>7.3f} {row['angular_error_deg']:>8.2f}")
print({'verdict': demo_report['verdict'], 'valid_pixels': demo_report['metrics']['valid_pixels']})

strip = Image.new('RGB', (4 * 256, 256))
for index, panel in enumerate((demo['image1'], demo['image2'], flow_to_rgb(demo['flow']), flow_to_rgb(demo_result['flow']))):
    strip.paste(panel.resize((256, 256)), (256 * index, 0))
print('frame 1 | frame 2 | true flow | predicted flow (hue = direction, saturation = speed)')
strip

**What to notice:** the RAFT row against the zero-flow row, the validity fraction, the mismatched-size rejection, and the four-panel strip.

<details><summary>Check your reasoning</summary>Zero flow scores the mean true displacement; in the recorded run that was EPE 6.8937 against 0.1625 for RAFT, with 98.8 % of valid pixels within 1 px. Integer shifts of textured frames are an easy case, which is why the zero-flow row, not 0, is the reference, and why one rendered pair is `sample-sanity`, not a benchmark.</details>

## 5. Recurrent updates and degenerate input probes

RAFT's estimate depends on how many times the update block refines it. torchvision's default is 12 updates, and the builder accepts any count; this pipeline allows 1 to 32. The first table re-scores the demonstration pair at 1, 4, 12 and 32 updates, so you can see where more refinement stops paying for its compute.

Then ask the model about pairs whose answer is known before trusting it:
- **Static pair:** the same textured frame twice. The true flow is zero everywhere, so any predicted motion is error.
- **Blank pair:** two featureless white frames. Motion is undetermined, because nothing can be matched. The cell reports the mean predicted displacement; a large value is motion the model invented.

**What to look for:** EPE close to zero on the static pair, and a small mean displacement on the blank pair.

**Predict before running:** will 32 updates always beat 12? And on two blank white frames, will the model report no motion at all?

In [ ]:
iteration_sweep = {}
for updates in (1, 4, 12, 32):
    flow = pipe.estimate(demo['image1'], demo['image2'], num_flow_updates=updates)['flow']
    iteration_sweep[updates] = {k: round(v, 4) for k, v in flow_metrics(flow, demo['flow'], demo['valid']).items() if k != 'valid_pixels'}
    print(f'{updates:>2d} updates', iteration_sweep[updates])

degenerate = {}
for probe in (static_pair(), blank_pair()):
    flow = pipe.estimate(probe['image1'], probe['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow']
    speed = np.linalg.norm(flow, axis=2)
    degenerate[probe['id']] = {'epe_vs_zero_truth': round(float(speed.mean()), 4), 'max_displacement_px': round(float(speed.max()), 4)}
print(json.dumps(degenerate, indent=2))

**What to notice:** the EPE at 1, 4, 12 and 32 updates, and the static and blank probe values.

<details><summary>Check your reasoning</summary>No and no. In the recorded run EPE fell from 0.4254 (1 update) to 0.1625 (12) but rose to 0.1893 at 32 — more refinement can drift. The static pair was near zero (EPE 0.0095), but the blank pair produced up to 0.8614 px of motion that does not exist: on featureless input the model invents flow.</details>

## 6. Labelled adaptation dataset and validation

`flow_dataset` renders a deterministic dataset of `N_PAIRS` pairs like the demonstration pair, each from its own seed, with background displacements up to 6 px and object displacements up to 12 px in each axis. Every record carries `image1`, `image2`, the exact `flow` and the `valid` mask.

`validate_dataset` checks every record before any model runs: the record keys, that both frames share one size inside the ceilings, that the flow has shape `(H, W, 2)` and is finite, and that the valid mask is boolean, has the frame's shape and selects at least one pixel. It returns a dataset manifest with the mean displacement and the mean valid fraction, and a finding for any valid pixel that moves `MAX_FLOW` (400) px or more, which the loss ignores.

In [ ]:
N_PAIRS = 40  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}
EPOCHS = 3  # @param {type:"integer"}

records = flow_dataset(N_PAIRS, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, epochs=EPOCHS)
print(json.dumps(dataset_manifest, indent=2))

preview = Image.new('RGB', (3 * 160, 2 * 160))
for index, record in enumerate(records[:3]):
    preview.paste(record['image1'].resize((160, 160)), (160 * index, 0))
    preview.paste(flow_to_rgb(record['flow']).resize((160, 160)), (160 * index, 160))
preview

## 7. Split and measure the pre-adaptation baseline

The dataset is split at random into a training part (`1 - HOLDOUT`) and a held-out part (`HOLDOUT`); with the defaults `N_PAIRS = 40` and `HOLDOUT = 0.25` that is 30 training and 10 held-out pairs, and the cell prints the actual counts. A random split is valid here because every pair is rendered independently. Pairs cut from the same video must be split by video instead, or the held-out frames will share content with the training frames. The held-out pairs are never shown to the optimizer, and no hyperparameter is selected on them.

The fine-tune runs on its own copy of the verified model, `adapter`, so `pipe` stays the pretrained reference. The baseline is that copy's held-out score before any update.

**The pretrained model already beats the zero-flow baseline here.** Textured frames moved by whole pixels are an easy case for RAFT, so a large gap between the two baseline rows is expected. The fine-tune has to improve on the RAFT row, not on zero flow.

*Evaluation practice.* **Predict before running:** why is a random split acceptable for these rendered pairs, when it would not be for frames cut from one video?

In [ ]:
HOLDOUT = 0.25  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

train_records, held_out = split_pairs(records, train_fraction=1.0 - HOLDOUT, seed=SEED)
overlap = {r['id'] for r in train_records} & {r['id'] for r in held_out}
assert not overlap, f'split leaked records: {sorted(overlap)}'
print({'train': len(train_records), 'held_out': len(held_out)})

adapter = RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
baseline = adapter.evaluate(held_out, num_flow_updates=NUM_FLOW_UPDATES)
METRIC_KEYS = ('epe', '1px', '3px', '5px', 'angular_error_deg')
print(json.dumps({'pretrained': {k: round(baseline[k], 4) for k in METRIC_KEYS},
                  'zero_flow': {k: round(baseline['zero_flow_baseline'][k], 4) for k in METRIC_KEYS}}, indent=2))

**What to notice:** the printed train / held-out counts (30 / 10 with the defaults), the pretrained row against the zero-flow row on the held-out pairs.

<details><summary>Check your reasoning</summary>Every rendered pair comes from its own seed, so no two pairs share content and a random split cannot leak. Frames from one video share scenes, so a random split there would test on near-copies of training frames. In the recorded run the held-out EPE was 0.1649 pretrained against 4.9648 for zero flow.</details>

## 8. Bounded fine-tuning

This cell runs the real adaptation step in this runtime. It is gradient fine-tuning with RAFT's **sequence loss**, the loss of the RAFT paper and of torchvision's reference training script. The model returns one flow estimate per recurrent update; the loss is the L1 distance to the true flow over valid pixels for each estimate, weighted by `gamma ** (N - i - 1)` with `gamma = 0.8`, so the last estimate weighs most. Pixels whose true displacement is 400 px or more are ignored, as in the reference.

- **The feature and context encoders are frozen by default.** With `FREEZE_ENCODERS = True` their weights keep the pretrained values, and only the recurrent update block and the upsampling mask predictor train. The cell prints the trainable and total parameter counts.
- **Every BatchNorm layer is held in evaluation mode**, so its running statistics do not drift either, as torchvision's reference does in its Sintel stage (`--freeze-batch-norm`).
- **Schedule:** `EPOCHS` epochs of AdamW at learning rate `2e-5` and weight decay `5e-5`, batch size 2, 12 recurrent updates, gradients clipped to norm 1.0, float32, seed `SEED`.

**Read the loss as optimisation evidence only.** A falling loss says the optimizer is fitting the training pairs; the held-out end-point error in the next section is the task evidence.

**Re-running this cell is safe.** It rebuilds `adapter` from the verified checkpoint before training, so each run is one fine-tune from the pretrained weights, never a second one stacked on already-adapted weights; the pretrained column in Section 9 still comes from Section 7, which scored an untouched copy.

**Predict before running:** with the encoders frozen, how many of the model's parameters will train?

In [ ]:
LEARNING_RATE = 2e-5  # @param {type:"number"}
BATCH_SIZE = 2  # @param {type:"integer"}
FREEZE_ENCODERS = True  # @param {type:"boolean"}

# Start from the verified checkpoint on every run of this cell (RF-M3), so changing a field and re-running it repeats
# one fine-tune instead of continuing from the weights the previous run adapted.
adapter = RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'adapter': 'rebuilt from the verified checkpoint', 'adapted': adapter.adapted})
run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_encoders=FREEZE_ENCODERS,
    progress=lambda row: print(f"epoch {row['epoch']}/{row['epochs']}  loss {row['loss']:.4f}"),
)
print(json.dumps({key: run[key] for key in ('freeze_encoders', 'batchnorm', 'trainable_parameters', 'total_parameters', 'epochs',
                                         'batch_size', 'learning_rate', 'optimizer', 'loss', 'precision', 'device', 'epoch_losses')}, indent=2))

**What to notice:** the trainable and total parameter counts, `freeze_encoders`, `batchnorm`, and the per-epoch losses.

<details><summary>Check your reasoning</summary>About three in five. In the recorded run 3,120,960 of 5,257,536 parameters trained; the loss fell 0.4394 → 0.3890 → 0.3614 over three epochs. A falling training loss is optimisation evidence only; Section 9 is the task evidence.</details>

## 9. Evaluate on the held-out split

`evaluate` re-runs on the same held-out pairs with the same number of updates as the baseline, so the rows are comparable. Each metric is the mean over pairs, each pair weighing the same; `pixel_weighted_epe` weighs every valid pixel the same instead. These are tutorial metrics from one pass over the held-out synthetic pairs (10 with the defaults), with no dispersion estimate. This notebook does not measure how far a different `SEED` moves these numbers, so a change of a few hundredths of a pixel may be within seed variation.

*Evaluation practice.* **Predict before running:** the pretrained held-out EPE is already about 0.16 px. Will the fine-tune lower it on every held-out pair?

In [ ]:
adapted = adapter.evaluate(held_out, num_flow_updates=NUM_FLOW_UPDATES)
print(f"{'metric':<18s} {'zero-flow':>10s} {'pretrained':>11s} {'adapted':>10s} {'change':>10s}")
for key in METRIC_KEYS + ('pixel_weighted_epe',):
    zero = baseline['zero_flow_baseline'][key]
    print(f"{key:<18s} {zero:>10.4f} {baseline[key]:>11.4f} {adapted[key]:>10.4f} {adapted[key] - baseline[key]:>+10.4f}")

**What to notice:** the change column for every metric, and `pixel_weighted_epe` beside the per-pair mean.

<details><summary>Check your reasoning</summary>In the recorded run, yes: held-out EPE went 0.1649 → 0.1446 and was lower on all 10 pairs (1 px rate 0.9855 → 0.9878). Ten rendered pairs and one seed carry no dispersion estimate, and the model was fitted to renderings from the same generator, so this says nothing about camera footage.</details>

**Checkpoint: is a gain of about 0.02 px real?** Before opening the answer, decide what you would need to see to believe it.

<details><summary>Check your reasoning</summary>Not established by this run. The gain (0.1649 → 0.1446, −0.020 px) is consistent — every held-out pair improved in the recorded run — but the notebook never measures how far a different `SEED` moves the scores, so the gain is stated against an *unmeasured* variation, and in Section 10 one of three unseen pairs got slightly worse (0.0976 → 0.0987). To measure it, set `SEED` to 1 and then 2 in Section 7, rerun Sections 7–9 each time, and compare the spread of the adapted EPE with the gain.</details>

### Activity: does unfreezing the encoders help? (Predict → Change → Run → Observe → Explain)

1. **Predict.** Write down whether held-out EPE will fall further, stay or rise when the feature and context encoders train too, and how `trainable_parameters` will change.
2. **Change.** In Section 8 set `FREEZE_ENCODERS = False`.
3. **Run.** Run the Section 8 cell, then the Section 9 cell. Section 8 rebuilds `adapter` from the verified checkpoint every time, so this is one fresh fine-tune with the new setting, not a second one stacked on the first. (To change `EPOCHS` or `N_PAIRS`, rerun from Section 6; to change `NUM_FLOW_UPDATES`, rerun from Section 4.)
4. **Observe.** Compare `trainable_parameters`, `epoch_losses` and the Section 9 change column with your first run.
5. **Explain.** Is the difference larger than the seed variation the Section 9 checkpoint asks about? Then set `FREEZE_ENCODERS = True` again and rerun Sections 8–12, so the exported adapter and the output files describe the default run.

<details><summary>Check your reasoning</summary>Unfrozen, all 5,257,536 parameters train instead of 3,120,960. In the review's local CPU run of this exercise (4 October 2026; that CPU run reproduced the recorded T4 frozen numbers to four decimals) held-out EPE fell to 0.1089 against 0.1446 frozen. The encoders were also fitted to these renderings, so a larger gain here is still a gain on the same synthetic generator, and the comparison carries no dispersion estimate. Your GPU numbers will differ in the last digits.</details>

## 10. Inference on unseen pairs

Three new pairs come from a seed the dataset never used (`NEW_DATA_SEED = 99`). The pretrained and the adapted pipelines estimate flow on each, and both are scored against the exact flow.

**Predict before running:** will the adapted model be better on all three unseen pairs?

In [ ]:
NEW_DATA_SEED = 99  # @param {type:"integer"}

new_records = flow_dataset(3, seed=NEW_DATA_SEED)
new_data_rows = []
for record in new_records:
    row = {'id': record['id'], 'mean_true_displacement_px': round(float(np.linalg.norm(record['flow'], axis=2)[record['valid']].mean()), 3)}
    for name, model in (('pretrained', pipe), ('adapted', adapter)):
        flow = model.estimate(record['image1'], record['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow']
        row[f'{name}_epe'] = round(flow_metrics(flow, record['flow'], record['valid'])['epe'], 4)
    new_data_rows.append(row)
    print(json.dumps(row))

**What to notice:** pretrained against adapted EPE for each unseen pair.

<details><summary>Check your reasoning</summary>No. In the recorded run two pairs improved (0.0978 → 0.0921, 0.2836 → 0.2161) and one got slightly worse (0.0976 → 0.0987). A held-out mean can improve while individual cases regress; report both.</details>

## 11. Adapter export, fresh reload, and equivalence check

`save_artifact` writes `outputs/raft_adapter.safetensors`: every tensor the fine-tune could change, plus a metadata header naming the base model, its pinned checkpoint SHA-256 and the frozen prefixes. With the encoders frozen, their tensors are left out because they equal the verified base checkpoint.

`load_artifact` then builds a **fresh** pipeline from the verified base checkpoint, loads the adapter tensors onto it, and refuses an adapter whose format, base identity, base digest or tensor set does not fit. The cell compares the reloaded flow with the in-memory model's on an unseen pair, with a stated tolerance: loading succeeding is not the check, reproducing the flow is.

In [ ]:
artifact_path = OUTPUTS / 'raft_adapter.safetensors'
descriptor = adapter.save_artifact(artifact_path, notes='RAFT-Large moving-shapes adaptation tutorial adapter')
print(json.dumps(descriptor, indent=2))

reloaded = RaftPipeline.load_artifact(artifact_path, weights_dir=WEIGHTS_DIR)
print({'reloaded_source': reloaded.source, 'adapted': reloaded.adapted, 'frozen_prefixes': list(reloaded.frozen_prefixes)})

TOLERANCE_PX = 1e-3
probe = new_records[0]
flow_orig = adapter.estimate(probe['image1'], probe['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow']
flow_reloaded = reloaded.estimate(probe['image1'], probe['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow']
difference = float(np.linalg.norm(flow_orig - flow_reloaded, axis=2).mean())
assert difference <= TOLERANCE_PX, f'reloaded flow differs by {difference:.6f} px on average'
reload_check = {'pair': probe['id'], 'mean_endpoint_difference_px': difference, 'tolerance_px': TOLERANCE_PX, 'equivalent': True}
print(reload_check)

**What to notice:** the adapter descriptor (tensor count, size, base digest) and `mean_endpoint_difference_px` against the tolerance.

<details><summary>Check your reasoning</summary>In the recorded run the 30-tensor, 12.5 MB adapter reloaded onto a fresh verified base and reproduced the flow exactly (mean difference 0.0 px against a 0.001 px tolerance). Loading is not the check; reproducing the flow is.</details>

## 12. Write machine-readable outputs and provenance

The cell writes:
- `outputs/raft_optical_flow_input_manifest.json`
- `outputs/raft_optical_flow_evaluation_report.json`
- `outputs/raft_optical_flow_result.json` (identity, runtime versions, device, iteration sweep, probes, dataset manifest, split, baseline and adapted metrics, fine-tuning configuration, new-data rows, adapter descriptor and reload check)
- `outputs/raft_optical_flow_metrics.csv` (one row per held-out and unseen pair, pretrained and adapted EPE)
- `outputs/raft_optical_flow_flow.flo` and `outputs/raft_optical_flow_flow.npy` (the demonstration pair's predicted flow)
- `outputs/raft_optical_flow_flow.png` (the same flow, colour-coded)
- `outputs/raft_adapter.safetensors` (written in Section 11)

In [ ]:
import csv

with open(OUTPUTS / 'raft_optical_flow_input_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(input_manifest, f, indent=2)

with open(OUTPUTS / 'raft_optical_flow_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(demo_report, f, indent=2)

write_flo(OUTPUTS / 'raft_optical_flow_flow.flo', demo_result['flow'])
np.save(OUTPUTS / 'raft_optical_flow_flow.npy', demo_result['flow'])
flow_to_rgb(demo_result['flow']).save(OUTPUTS / 'raft_optical_flow_flow.png')
assert np.array_equal(read_flo(OUTPUTS / 'raft_optical_flow_flow.flo'), demo_result['flow'])

with open(OUTPUTS / 'raft_optical_flow_metrics.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['split', 'pair', 'pretrained_epe', 'adapted_epe', 'zero_flow_epe'])
    for split, group in (('held_out', held_out), ('unseen', new_records)):
        for record in group:
            scores = [flow_metrics(m.estimate(record['image1'], record['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow'], record['flow'], record['valid'])['epe'] for m in (pipe, adapter)]
            zero = flow_metrics(np.zeros_like(record['flow']), record['flow'], record['valid'])['epe']
            writer.writerow([split, record['id'], *(f'{v:.4f}' for v in (*scores, zero))])

result_export = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'weights_url': WEIGHTS_URL,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
                'cuda': torch.cuda.is_available()},
    'device': pipe.device,
    'num_flow_updates': NUM_FLOW_UPDATES,
    'demo_metrics': demo_report['metrics'],
    'iteration_sweep': iteration_sweep,
    'degenerate_probes': degenerate,
    'adaptation': {
        'dataset': dataset_manifest,
        'dataset_seed': DATASET_SEED,
        'split': {'train': len(train_records), 'held_out': len(held_out), 'seed': SEED, 'holdout': HOLDOUT},
        'finetune': run,
        'baseline': baseline,
        'adapted': adapted,
        'new_data': new_data_rows,
        'artifact': descriptor,
        'reload_check': reload_check,
    },
}
with open(OUTPUTS / 'raft_optical_flow_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_export, f, indent=2, default=str)

for p in sorted(OUTPUTS.iterdir()):
    if p.is_file():
        print(f'  {p.name:<44s} {p.stat().st_size:>12,d} bytes')

## 13. Optional: Bring Your Own Data (BYOD)

Both branches are off by default, so `Run all` never stops here. Before you turn one on, read the contract:

- **Frame-pair branch** (`USE_BYOD_IMAGE`): two image files PIL can open, of the same size, each side between `MIN_IMAGE_SIDE` (128) and `MAX_IMAGE_SIDE` (1024) px and at most `MAX_PIXELS` (786,432) pixels. Frame 1 comes before frame 2 in time. The pair runs through `validate_inputs`, `estimate` and `evaluation_report`; the report is `not-measurable`, because no ground-truth flow comes with it.
- **Dataset branch** (`USE_BYOD_DATASET`): a directory with `pairs.json` (a list of `{'frame1', 'frame2', 'flow', 'valid'}` objects) and the files it names, at least 2 records and at most 2,000. `flow` is a Middlebury `.flo` file or a `.npy` array of shape `(H, W, 2)`; `valid` is an optional single-channel PNG whose non-zero pixels are valid. File names must stay inside the directory. The branch runs the same validate → split → baseline → fine-tune → evaluate → export → reload stages as the sample, checks that the reloaded adapter reproduces the flow on a held-out pair, and writes the zero-flow, pretrained and adapted metrics, the split sizes and the reload check to `outputs/byod_result.json`. With fewer than 5 held-out pairs it prints a small-split warning: the documented minimum of 2 records leaves 1 held-out pair, which is an anecdote, not an estimate.

Set `BYOD_IMAGE1_PATH` and `BYOD_IMAGE2_PATH`, or `BYOD_DATASET_DIR`, to read from a mounted or local location; leave them empty on Colab to get an upload dialog instead (for the pair branch, upload both frames at once; they are ordered by file name). Uploaded files are written under `outputs/byod/` in this runtime and are not sent anywhere else. The first lines of the cell show the validator refusing two malformed inputs with messages that name the failed rule.

In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
BYOD_IMAGE1_PATH = ""  # @param {type:"string"}
BYOD_IMAGE2_PATH = ""  # @param {type:"string"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_DATASET_DIR = ""  # @param {type:"string"}

_probe = static_pair()
for desc, probe_call in (
    ('non-image object', lambda: validate_inputs('/not/an/image.png', _probe['image2'])),
    ('flow of the wrong shape', lambda: validate_dataset([{'image1': _probe['image1'], 'image2': _probe['image2'],
                                                          'flow': np.zeros((8, 8, 2), np.float32)}])),
):
    try:
        probe_call()
    except (TypeError, ValueError) as exc:
        print(f'refused as expected: {desc} -> {type(exc).__name__}: {exc}')

BYOD_DIR = OUTPUTS / 'byod'
BYOD_MIN_HELD_OUT = 5  # fewer held-out pairs than this prints a small-split warning

def _upload_into(target, fields):
    try:
        from google.colab import files  # type: ignore[import-not-found]
    except ImportError:
        raise ValueError(f'The upload dialog exists only on Google Colab. Set {fields} to a local or mounted path '
                         'and run this cell again.') from None
    target.mkdir(parents=True, exist_ok=True)
    for name, data in files.upload().items():
        (target / Path(name).name).write_bytes(data)
    return target

if USE_BYOD_IMAGE:
    if BYOD_IMAGE1_PATH and BYOD_IMAGE2_PATH:
        frame_paths = [Path(BYOD_IMAGE1_PATH), Path(BYOD_IMAGE2_PATH)]
    else:
        frame_paths = sorted(p for p in _upload_into(BYOD_DIR / 'pair', 'BYOD_IMAGE1_PATH and BYOD_IMAGE2_PATH').iterdir() if p.is_file())[:2]
    if len(frame_paths) != 2:
        raise ValueError(f'the frame-pair branch needs exactly two frames, got {len(frame_paths)}')
    byod_frames = []
    for frame_path in frame_paths:
        with Image.open(frame_path) as handle:
            byod_frames.append(handle.convert('RGB'))
    print(validate_inputs(*byod_frames, num_flow_updates=NUM_FLOW_UPDATES)['verdict'], [p.name for p in frame_paths])
    byod_result = pipe.estimate(*byod_frames, num_flow_updates=NUM_FLOW_UPDATES)
    byod_report = evaluation_report(byod_result, sample_kind='byod')
    print({'verdict': byod_report['verdict'], 'mean_predicted_displacement_px': round(byod_report['mean_predicted_displacement_px'], 3)})
    write_flo(OUTPUTS / 'byod_flow.flo', byod_result['flow'])
    flow_to_rgb(byod_result['flow']).save(OUTPUTS / 'byod_flow.png')
else:
    print('BYOD frame-pair branch is off; set USE_BYOD_IMAGE = True to estimate flow between your own two frames.')

if USE_BYOD_DATASET:
    dataset_dir = Path(BYOD_DATASET_DIR) if BYOD_DATASET_DIR else _upload_into(BYOD_DIR / 'dataset', 'BYOD_DATASET_DIR')
    byod_records = read_flow_records(dataset_dir)
    byod_manifest = validate_dataset(byod_records, epochs=EPOCHS)
    print(json.dumps(byod_manifest, indent=2))
    byod_train, byod_held = split_pairs(byod_records, train_fraction=1.0 - HOLDOUT, seed=SEED)
    print({'train': len(byod_train), 'held_out': len(byod_held)})
    if len(byod_held) < BYOD_MIN_HELD_OUT:
        print(f'WARNING: only {len(byod_held)} held-out pair(s). The scores below are anecdotes, not an estimate; '
              f'supply enough pairs for at least {BYOD_MIN_HELD_OUT} held-out pairs before drawing a conclusion.')
    byod_pipe = RaftPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
    byod_baseline = byod_pipe.evaluate(byod_held, num_flow_updates=NUM_FLOW_UPDATES)
    byod_run = byod_pipe.finetune(byod_train, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED, freeze_encoders=FREEZE_ENCODERS)
    byod_adapted = byod_pipe.evaluate(byod_held, num_flow_updates=NUM_FLOW_UPDATES)
    print(f"{'metric':<18s} {'zero-flow':>10s} {'pretrained':>11s} {'adapted':>10s} {'change':>10s}")
    for key in METRIC_KEYS + ('pixel_weighted_epe',):
        zero = byod_baseline['zero_flow_baseline'][key]
        print(f"{key:<18s} {zero:>10.4f} {byod_baseline[key]:>11.4f} {byod_adapted[key]:>10.4f} {byod_adapted[key] - byod_baseline[key]:>+10.4f}")
    byod_artifact = OUTPUTS / 'byod_raft_adapter.safetensors'
    byod_descriptor = byod_pipe.save_artifact(byod_artifact, notes='BYOD adaptation adapter')
    byod_reloaded = RaftPipeline.load_artifact(byod_artifact, weights_dir=WEIGHTS_DIR)
    byod_probe = byod_held[0]
    byod_flows = [m.estimate(byod_probe['image1'], byod_probe['image2'], num_flow_updates=NUM_FLOW_UPDATES)['flow'] for m in (byod_pipe, byod_reloaded)]
    byod_difference = float(np.linalg.norm(byod_flows[0] - byod_flows[1], axis=2).mean())
    assert byod_difference <= TOLERANCE_PX, f'reloaded BYOD flow differs by {byod_difference:.6f} px on average'
    byod_reload_check = {'pair': byod_probe['id'], 'mean_endpoint_difference_px': byod_difference, 'tolerance_px': TOLERANCE_PX, 'equivalent': True}
    print(byod_reload_check)
    byod_export = {
        'notebook_source': NOTEBOOK_SOURCE, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'sample_kind': 'byod',
        'dataset': byod_manifest,
        'split': {'train': len(byod_train), 'held_out': len(byod_held), 'seed': SEED, 'holdout': HOLDOUT,
                  'small_split_warning': len(byod_held) < BYOD_MIN_HELD_OUT},
        'finetune': byod_run, 'baseline': byod_baseline, 'adapted': byod_adapted,
        'artifact': byod_descriptor, 'reload_check': byod_reload_check,
    }
    with open(OUTPUTS / 'byod_result.json', 'w', encoding='utf-8') as f:
        json.dump(byod_export, f, indent=2, default=str)
    print('BYOD adapter exported, reloaded and checked; results in outputs/byod_result.json:', byod_descriptor['sha256'][:16])
else:
    print('BYOD dataset branch is off; set USE_BYOD_DATASET = True to adapt RAFT on your own frame pairs with ground-truth flow.')

## Interpretation and limits

**What this notebook established, in this runtime.** The pinned torchvision checkpoint for `torchvision/raft_large` was verified against a committed SHA-256 manifest before loading. The pretrained model estimated flow on a rendered pair with exact ground truth, was re-scored at four update counts, and was probed with a static and a featureless pair. A labelled dataset of rendered pairs was then used to fine-tune the recurrent update block with RAFT's sequence loss, with the encoders and every BatchNorm layer frozen, and the held-out split was scored by end-point error before and after, next to the zero-flow baseline. The adapter was exported, reloaded onto a fresh base model and checked against the in-memory flow.

**How large the gain is.** With the defaults the recorded run moved held-out EPE from 0.1649 to 0.1446 px (−0.020 px), lower on every held-out pair, while one of three unseen pairs got slightly worse (0.0976 → 0.0987). Seed variation was not measured, so this notebook cannot say whether a gain of this size exceeds what a different `SEED` would produce; Section 9's checkpoint shows how to measure it.

**What a green run proves.** Successful execution proves that the recorded repository revision, the pinned dependency set and the pinned checkpoint together reproduce these stages in a fresh runtime, without the repository being cloned or installed and without any DIMER worker or service. It does **not** establish benchmark superiority, fitness for any deployment, or that the adapted model generalises beyond the rendered pairs it was fitted to. The held-out EPE is measured on 10 rendered pairs with the default `N_PAIRS` and `HOLDOUT` and carries no dispersion estimate.

**Reproducibility.** Seeds are form fields (`DEMO_SEED`, `DATASET_SEED`, `SEED`, `NEW_DATA_SEED`), and the run is float32 with no data augmentation. GPU kernels (the correlation lookup's `grid_sample` backward pass in particular) are not forced to be deterministic, so repeated GPU runs can differ in the last digits of the loss and the scores.

**Try next.** The activity after Section 9 compares `FREEZE_ENCODERS = False` with the default. To change `NUM_FLOW_UPDATES` and watch the EPE and the time per pair, set it in Section 4 and rerun from Section 4. To transfer the workflow, point `BYOD_DATASET_DIR` at frame pairs with ground-truth flow from your own domain, for example rendered from a simulator.

## Troubleshooting

- **Section 1 stops with "This notebook needs a Linux x86_64 runtime"** — you are on Windows, macOS or an ARM machine. Use Google Colab, Kaggle or a Linux x86_64 Jupyter server.
- **The uv wheel fails its size/SHA-256 check, or a download in Section 1 times out** — run Section 1 again; a complete environment built from the same lock is reused, an incomplete one is finished. If it repeats, the network is blocking or altering `files.pythonhosted.org` or `pypi.org`.
- **"The isolated environment's Python process exited"** — usually out of memory. Restart the session and choose **Run all**.
- **You re-ran Section 1 on its own** — nothing is lost: it keeps the running worker and every variable, so the cells after it keep working. After a session restart, run from the top.
- **Section 3 reports a size or SHA-256 mismatch, or cannot reach the Hub** — the message names the file. Delete it from the snapshot folder Section 3 prints and run Section 3 again.
- **Section 3 reports a size or SHA-256 mismatch for the checkpoint** — the download from `download.pytorch.org` was cut short or altered; delete the file in the snapshot folder and run Section 3 again.
- **The fine-tune is slow** — you are on CPU; switch to a T4 GPU.
- **Your numbers differ in the last digits from the recorded run** — GPU kernels (the correlation lookup's backward pass) are not deterministic; the comparison between rows is the result.
- **BYOD: a frame-size, `pairs.json` or flow-shape refusal** — the message names the rule; frames must share one size inside the ceilings, and flow must be a `.flo` or `(H, W, 2)` `.npy` file. Set `BYOD_IMAGE1_PATH`/`BYOD_IMAGE2_PATH` or `BYOD_DATASET_DIR` to skip the upload dialog.

## Glossary

- **Optical flow:** the per-pixel displacement `(u, v)` that maps each point of frame 1 to where it appears in frame 2.
- **Valid mask:** the pixels whose destination is visible in frame 2; only these are scored.
- **EPE (end-point error):** the mean distance in pixels between predicted and true displacement; lower is better.
- **1px / 3px / 5px:** the share of valid pixels whose error is under that many pixels.
- **Angular error:** the mean angle between the space-time vectors `(u, v, 1)` of prediction and truth.
- **Zero-flow baseline:** predicting no motion anywhere; its EPE equals the mean true displacement.
- **Recurrent updates:** RAFT refines its estimate a fixed number of times; 12 by default.
- **Sequence loss:** RAFT's training loss: the L1 error of every intermediate estimate, later ones weighted more.
- **Frozen encoders / BatchNorm:** the feature and context encoders keep their pretrained weights, and normalisation statistics do not drift.
- **Adapter:** the SafeTensors file holding only the tensors the fine-tune changed, reloaded onto the verified base.
- **Isolated environment:** the separate Python environment Section 1 builds from the hash lock; every later cell runs there.

## Conclusion (your notes)

Complete these in your own words; the recorded run's values are in the **Check your reasoning** answers above.

- On the demonstration pair RAFT scored EPE ___ against zero flow's ___; the blank-pair probe showed ___.
- The fine-tune moved held-out EPE from ___ to ___; on unseen pairs ___.
- The number I would not trust on its own is ___, because ___.
- Before adapting on my own footage I would split by ___ and compare against ___.

## References

- Teed, Z. and Deng, J. (2020). *RAFT: Recurrent All-Pairs Field Transforms for Optical Flow.* ECCV 2020. [arXiv:2003.12039](https://arxiv.org/abs/2003.12039).
- torchvision model documentation: [raft_large](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.optical_flow.raft_large.html); upstream repository [pytorch/vision](https://github.com/pytorch/vision) — BSD-3-Clause; training recipe [references/optical_flow](https://github.com/pytorch/vision/tree/main/references/optical_flow).
- Butler, D. J., Wulff, J., Stanley, G. B. and Black, M. J. (2012). *A Naturalistic Open Source Movie for Optical Flow Evaluation.* ECCV 2012 — the Sintel benchmark.
- Baker, S., Scharstein, D., Lewis, J. P., Roth, S., Black, M. J. and Szeliski, R. (2011). *A Database and Evaluation Methodology for Optical Flow.* IJCV 92 — the Middlebury `.flo` format and the angular error.
- Repository model card: https://github.com/kurtvalcorza/raft-optical-flow-pipeline/blob/main/MODEL_CARD.md
- [`kurtvalcorza/raft-optical-flow-pipeline`](https://github.com/kurtvalcorza/raft-optical-flow-pipeline) — source repository for this pipeline.